# Coherent deferred-measurement teleportation

Use controlled corrections instead of native mid-circuit control and compare receiver observables.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

Deferred measurement rewrites measurement-conditioned operations into an equivalent unitary circuit suitable for this backend.

In [2]:
def make_qnode(device):
    @qml.qnode(device)
    def teleport(theta):
        qml.RY(theta, wires=0)
        qml.Hadamard(1)
        qml.CNOT(wires=[1, 2])
        qml.CNOT(wires=[0, 1])
        qml.Hadamard(0)
        qml.CNOT(wires=[1, 2])
        qml.CZ(wires=[0, 2])
        return qml.expval(qml.X(2)), qml.expval(qml.Z(2))
    return teleport

angles = np.linspace(-1.1, 1.1, 9)
reference_qnode = make_qnode(qml.device("default.qubit", wires=3))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: np.asarray([reference_qnode(value) for value in angles]))

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=3, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: np.asarray([mettleq_qnode(value) for value in angles]))
error = max_abs_error(reference, candidate)
analytic = np.column_stack([np.sin(angles), np.cos(angles)])
analytic_error = max_abs_error(analytic, candidate)
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

Receiver observables are compared across several input states.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/10_teleportation_deferred.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="receiver observables atol=3e-6",
    passed=max(error, analytic_error) <= 3e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"reference_error": error, "analytic_receiver_error": analytic_error},
    notes="MettleQ does not claim native mid-circuit control; this notebook applies the deferred coherent circuit explicitly.",
)


Comparison summary
------------------
Correctness contract: PASS — receiver observables atol=3e-6
SDK reference median: 7.967 ms
MettleQ median:       26.814 ms
Timing interpretation: the SDK reference was 3.366x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)
Note: MettleQ does not claim native mid-circuit control; this notebook applies the deferred coherent circuit explicitly.

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "receiver observables atol=3e-6", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"analytic_receiver_error": 1.819067652064632e-07, "reference_error": 1.819067652064632e-07}, "mettleq_median_ms": 26.814250013558194, "notebook": "pennylane/10_teleportation_deferred.ipynb", "notes": "MettleQ does not claim native mid-circuit control; this notebook applies the deferred coherent circuit explicitly.", "passed": true, 

## What should you conclude?

MettleQ currently targets the deferred unitary form, not native mid-circuit classical control.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.